# Data Analyzer Comparison: Raw vs Template vs Parameterized (Real API)

Compare three approaches using the **actual `DataAnalyzer` API** (`intent` + `investment` parameters):
1. **Raw** — generic prompt (no structured template)
2. **Template** — `DataAnalyzer(intent="comprehensive")` (all 11 sections)
3. **Parameterized** — `DataAnalyzer(intent=..., investment=...)` (executive, analyst, summary)

Evidence tracked: output length (chars, approx tokens), section coverage, quality scores (LLM-as-judge).

**Requires**: `OPENAI_API_KEY` for LLM runs. Set `EXECUTE_LLM = True` to call the API.

In [1]:
%load_ext autoreload
%autoreload 2

import io
import os
from pathlib import Path

import pandas as pd

os.environ.setdefault('OPENAI_API_KEY', 'sk-...')  # Set your key here or via environment variable
os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
os.environ['GOOGLE_API_KEY'] = 'AIza...'

# Toggle: False = build context only (no API calls); True = run LLM
EXECUTE_LLM = True

# Paths (run from repo root, docs/examples, or examples/)
CSV_PATH = Path("sample_analysis_data.csv")
for p in [Path("../sample_analysis_data.csv"), Path("examples/sample_analysis_data.csv"), Path("docs/examples/sample_analysis_data.csv")]:
    if p.exists():
        CSV_PATH = p
        break

C:\Users\dpokh\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 1. Build data description from CSV

Use pandas to create a structured description (no extra LLM call).

In [2]:
df = pd.read_csv(CSV_PATH)
buf = io.StringIO()
df.info(buf=buf)

data_description = f"""Columns: {list(df.columns)}
Shape: {df.shape[0]} rows, {df.shape[1]} columns

Info:
{buf.getvalue()}

Describe:
{df.describe().to_string()}

Sample (first 8 rows):
{df.head(8).to_string()}
"""

goal = "Identify growth opportunities, regional differences, and anomalies. Give 3-5 key insights with actions."
context_extra = "Monthly revenue and units by region (North, South, East) and product (Widget A, B). Cost is COGS."

print("Data description:", len(data_description), "chars")
print("Goal:", goal[:80], "...")
df.head(5)

Data description: 1458 chars
Goal: Identify growth opportunities, regional differences, and anomalies. Give 3-5 key ...


,month,region,product,revenue,units,cost
0,2024-01,North,Widget A,12400,310,8200
1,2024-01,North,Widget B,8900,178,5200
2,2024-01,South,Widget A,15200,380,10100
3,2024-01,South,Widget B,7600,152,4500
4,2024-01,East,Widget A,9800,245,6600


## 2. DataAnalyzer API: intent + investment

Uses the real `DataAnalyzer` class with `intent` and `investment` parameters.
Available intents: `executive`, `analyst`, `operations`, `summary`, `comprehensive` (default).
Available investments: `quick`, `standard` (default), `thorough`.

In [3]:
from mycontext.templates.free.analysis import DataAnalyzer

analyzer = DataAnalyzer()

print("Available intents:", sorted(analyzer.INTENTS.keys()))
print("Available investments:", sorted(analyzer.INVESTMENTS.keys()))
print()

for intent_name, sections in analyzer.INTENTS.items():
    label = "all 11 sections" if sections is None else ", ".join(sections)
    print(f"  {intent_name}: {label}")

ALL_SECTIONS = [
    "DATA OVERVIEW", "DESCRIPTIVE STATISTICS", "PATTERN DETECTION",
    "ANOMALY DETECTION", "CORRELATION ANALYSIS", "COMPARATIVE ANALYSIS",
    "KEY INSIGHTS", "HYPOTHESES", "DATA LIMITATIONS",
    "RECOMMENDATIONS", "VISUALIZATION SUGGESTIONS",
]

Available intents: ['analyst', 'comprehensive', 'executive', 'operations', 'summary']
Available investments: ['quick', 'standard', 'thorough']

  executive: data_overview, key_insights, visualization_suggestions, recommendations
  analyst: data_overview, descriptive_statistics, pattern_detection, correlation_analysis
  operations: data_overview, anomaly_detection, recommendations
  summary: data_overview, key_insights, recommendations
  comprehensive: all 11 sections


## 3. Run all modes via real DataAnalyzer API

Execute Raw (generic prompt) and four intent+investment combinations using `DataAnalyzer.execute()`.

In [4]:
import inspect
from mycontext.providers.litellm_provider import LiteLLMProvider, _reasoning_tokens_exhausted

src = inspect.getsource(LiteLLMProvider.generate)
print("Provider checks:")
print(f"  - Fallback user message:       {'effective_user' in src}")
print(f"  - Reasoning model retry:       {'_reasoning_tokens_exhausted' in src}")
print("Ready to run.")

Provider checks:
  - Fallback user message:       True
  - Reasoning model retry:       True
Ready to run.


## 3a. Model Comparison: Reasoning vs Lightweight

**Hypothesis**: If the template provides enough structure (sections, priorities, constraints),
a lightweight model (gpt-4o-mini, ~$0.15/M tokens) should deliver comparable quality
to a reasoning model (gpt-5.2, ~$5+/M tokens) — without the hidden reasoning-token overhead.

Tests:
- **Raw prompt** on both models (no template guidance)
- **Template (executive intent)** on both models (structured guidance)
- **Template (comprehensive)** on both models

Tracked: output quality, token usage, cost, section coverage.

In [ ]:
from mycontext import Context
from mycontext.foundation import Directive

MODELS = {
    "gpt-4o-mini": {"label": "lightweight", "reasoning": False},
    "gpt-4o":      {"label": "mid-tier",    "reasoning": False},
    "gpt-5.2":     {"label": "reasoning",   "reasoning": True},
}

SCENARIOS = [
    {"name": "raw",          "intent": None,            "investment": None},
    {"name": "executive",    "intent": "executive",     "investment": "standard"},
    {"name": "comprehensive","intent": "comprehensive", "investment": "standard"},
]


def _extract(result) -> str:
    if hasattr(result, "response"):
        return result.response or ""
    return str(result)


def run_scenario(model: str, scenario: dict) -> dict:
    """Run a single (model × scenario) combination and return metrics."""
    name = scenario["name"]

    if name == "raw":
        prompt = analyzer.generic_prompt(
            data_description=data_description,
            goal=goal,
            context_section=f"\nContext: {context_extra}" if context_extra else "",
        )
        ctx = Context(directive=Directive(content=prompt))
        result = ctx.execute(provider="openai", model=model, use_cache=False)
    else:
        result = analyzer.execute(
            provider="openai",
            model=model,
            data_description=data_description,
            goal=goal,
            context=context_extra,
            intent=scenario["intent"],
            investment=scenario["investment"],
            use_cache=False,
        )

    out = _extract(result)
    meta = result.metadata if hasattr(result, "metadata") else {}

    upper = out.upper()
    sections_hit = sum(1 for s in ALL_SECTIONS if s in upper)

    return {
        "model": model,
        "model_type": MODELS[model]["label"],
        "scenario": name,
        "output_chars": len(out),
        "output_tokens_approx": len(out) // 4,
        "input_tokens": meta.get("input_tokens", 0),
        "output_tokens": meta.get("output_tokens", 0),
        "total_tokens": result.tokens_used if hasattr(result, "tokens_used") else 0,
        "cost_usd": result.cost_usd if hasattr(result, "cost_usd") else 0,
        "finish_reason": meta.get("finish_reason", ""),
        "sections_found": sections_hit,
        "full_output": out,
    }


print(f"Models:    {list(MODELS.keys())}")
print(f"Scenarios: {[s['name'] for s in SCENARIOS]}")
print(f"Total runs: {len(MODELS) * len(SCENARIOS)}")

In [ ]:
model_results = []
total = len(MODELS) * len(SCENARIOS)
i = 0

for model_name in MODELS:
    for scenario in SCENARIOS:
        i += 1
        print(f"{i}/{total}  {model_name} × {scenario['name']}...", end=" ", flush=True)
        try:
            r = run_scenario(model_name, scenario)
            model_results.append(r)
            print(f"✓  {r['output_chars']} chars, {r['total_tokens']} tokens, ${r['cost_usd']:.4f}")
        except Exception as e:
            print(f"✗  {e}")
            model_results.append({
                "model": model_name,
                "model_type": MODELS[model_name]["label"],
                "scenario": scenario["name"],
                "output_chars": 0, "output_tokens_approx": 0,
                "input_tokens": 0, "output_tokens": 0,
                "total_tokens": 0, "cost_usd": 0,
                "finish_reason": f"ERROR: {e}",
                "sections_found": 0, "full_output": "",
            })

print(f"\nDone — {len(model_results)} runs complete.")

In [ ]:
import pandas as pd

df_model = pd.DataFrame([
    {k: v for k, v in r.items() if k != "full_output"}
    for r in model_results
])

display_cols = [
    "model", "model_type", "scenario",
    "output_chars", "sections_found",
    "input_tokens", "output_tokens", "total_tokens",
    "cost_usd", "finish_reason",
]
df_model[display_cols].style.format({"cost_usd": "${:.4f}"}).background_gradient(
    subset=["cost_usd"], cmap="RdYlGn_r"
).background_gradient(
    subset=["sections_found"], cmap="Greens"
).background_gradient(
    subset=["output_chars"], cmap="Blues"
)

In [ ]:
print("=" * 70)
print("KEY FINDINGS: Model × Template Interaction")
print("=" * 70)

for scenario_name in ["raw", "executive", "comprehensive"]:
    subset = df_model[df_model["scenario"] == scenario_name].sort_values("cost_usd")
    print(f"\n--- {scenario_name.upper()} ---")
    for _, row in subset.iterrows():
        reasoning_flag = " (reasoning)" if MODELS[row["model"]]["reasoning"] else ""
        print(
            f"  {row['model']:15s}{reasoning_flag:14s} | "
            f"{row['output_chars']:6d} chars | "
            f"{row['sections_found']:2d}/11 sections | "
            f"{row['total_tokens']:5d} tokens | "
            f"${row['cost_usd']:.4f}"
        )

# Cost savings summary
print("\n" + "=" * 70)
print("COST COMPARISON (same scenario, cheapest vs most expensive)")
print("=" * 70)
for scenario_name in ["executive", "comprehensive"]:
    subset = df_model[df_model["scenario"] == scenario_name]
    if len(subset) < 2:
        continue
    cheapest = subset.loc[subset["cost_usd"].idxmin()]
    priciest = subset.loc[subset["cost_usd"].idxmax()]
    if priciest["cost_usd"] > 0:
        savings = (1 - cheapest["cost_usd"] / priciest["cost_usd"]) * 100
        section_diff = cheapest["sections_found"] - priciest["sections_found"]
        print(
            f"\n  {scenario_name}: {cheapest['model']} vs {priciest['model']}\n"
            f"    Cost:     ${cheapest['cost_usd']:.4f} vs ${priciest['cost_usd']:.4f}  "
            f"({savings:.0f}% cheaper)\n"
            f"    Sections: {cheapest['sections_found']} vs {priciest['sections_found']}  "
            f"({'same' if section_diff == 0 else f'{section_diff:+d}'})\n"
            f"    Tokens:   {cheapest['total_tokens']} vs {priciest['total_tokens']}"
        )

## 3b. Parameterization Quality Analysis

Does focusing on fewer sections (intent=executive, 4 sections) maintain quality
compared to asking for everything (intent=comprehensive, 11 sections)?

We measure:
- **Depth per section** — chars allocated per section (more = deeper analysis)
- **Section hit rate** — did the model produce exactly the intended sections?
- **Content comparison** — same section extracted from comprehensive vs focused

In [ ]:
import pandas as pd

INTENT_EXPECTED = {
    "executive":     ["DATA OVERVIEW", "KEY INSIGHTS", "VISUALIZATION SUGGESTIONS", "RECOMMENDATIONS"],
    "comprehensive": ALL_SECTIONS,
    "raw":           [],
}

rows = []
for r in model_results:
    scenario = r["scenario"]
    expected = INTENT_EXPECTED.get(scenario, [])
    expected_count = len(expected)
    found = r["sections_found"]
    chars = r["output_chars"]

    depth_per_section = round(chars / found, 0) if found > 0 else 0
    hit_rate = round(found / expected_count * 100, 1) if expected_count > 0 else None

    rows.append({
        "model": r["model"],
        "scenario": scenario,
        "expected_sections": expected_count,
        "sections_found": found,
        "hit_rate_%": hit_rate,
        "output_chars": chars,
        "depth_per_section": int(depth_per_section),
        "cost_usd": r["cost_usd"],
        "cost_per_section": round(r["cost_usd"] / found, 4) if found > 0 else 0,
    })

df_quality = pd.DataFrame(rows)
df_quality.style.format({
    "cost_usd": "${:.4f}",
    "cost_per_section": "${:.4f}",
    "hit_rate_%": "{:.0f}%",
}).background_gradient(
    subset=["depth_per_section"], cmap="YlOrRd"
).background_gradient(
    subset=["cost_per_section"], cmap="RdYlGn_r"
)

In [ ]:
print("=" * 75)
print("DEPTH ANALYSIS: Does parameterization degrade or improve section quality?")
print("=" * 75)

for model_name in MODELS:
    comp = [r for r in model_results if r["model"] == model_name and r["scenario"] == "comprehensive"]
    exec_ = [r for r in model_results if r["model"] == model_name and r["scenario"] == "executive"]
    raw = [r for r in model_results if r["model"] == model_name and r["scenario"] == "raw"]

    if not (comp and exec_ and raw):
        continue

    c, e, rw = comp[0], exec_[0], raw[0]
    c_depth = c["output_chars"] / c["sections_found"] if c["sections_found"] else 0
    e_depth = e["output_chars"] / e["sections_found"] if e["sections_found"] else 0
    rw_depth = rw["output_chars"] / rw["sections_found"] if rw["sections_found"] else 0

    depth_change = ((e_depth / c_depth) - 1) * 100 if c_depth > 0 else 0

    print(f"\n  {model_name} ({MODELS[model_name]['label']}):")
    print(f"    {'Scenario':<20s} {'Sections':>10s} {'Chars':>8s} {'Depth/Sect':>12s} {'Cost':>10s}")
    print(f"    {'-'*60}")
    print(f"    {'raw':<20s} {rw['sections_found']:>10d} {rw['output_chars']:>8d} {rw_depth:>12.0f} ${rw['cost_usd']:>9.4f}")
    print(f"    {'executive (focused)':<20s} {e['sections_found']:>10d} {e['output_chars']:>8d} {e_depth:>12.0f} ${e['cost_usd']:>9.4f}")
    print(f"    {'comprehensive':<20s} {c['sections_found']:>10d} {c['output_chars']:>8d} {c_depth:>12.0f} ${c['cost_usd']:>9.4f}")
    print(f"    → Focused vs Comprehensive depth: {depth_change:+.0f}% per section")

print("\n" + "=" * 75)
print("VERDICT")
print("=" * 75)

# Aggregate across models
all_comp_depths = []
all_exec_depths = []
for model_name in MODELS:
    c = [r for r in model_results if r["model"] == model_name and r["scenario"] == "comprehensive"]
    e = [r for r in model_results if r["model"] == model_name and r["scenario"] == "executive"]
    if c and e and c[0]["sections_found"] > 0 and e[0]["sections_found"] > 0:
        all_comp_depths.append(c[0]["output_chars"] / c[0]["sections_found"])
        all_exec_depths.append(e[0]["output_chars"] / e[0]["sections_found"])

avg_comp = sum(all_comp_depths) / len(all_comp_depths) if all_comp_depths else 0
avg_exec = sum(all_exec_depths) / len(all_exec_depths) if all_exec_depths else 0
avg_change = ((avg_exec / avg_comp) - 1) * 100 if avg_comp > 0 else 0

print(f"\n  Average depth per section:")
print(f"    Comprehensive: {avg_comp:.0f} chars/section")
print(f"    Executive:     {avg_exec:.0f} chars/section  ({avg_change:+.0f}%)")
if avg_change > 0:
    print(f"\n  ✓ Parameterization IMPROVES depth — focused intent gives {avg_change:.0f}%")
    print(f"    more content per section because the model concentrates its output.")
else:
    print(f"\n  ✗ Parameterization reduces depth by {abs(avg_change):.0f}% per section.")

In [ ]:
import re

def extract_section(text: str, section_name: str, all_sections: list) -> str:
    """Extract content of a specific section from the full output."""
    upper = text.upper()
    idx = upper.find(section_name)
    if idx == -1:
        return "(section not found)"

    start = idx
    # Find end: next section header or end of text
    end = len(text)
    for other in all_sections:
        if other == section_name:
            continue
        other_idx = upper.find(other, start + len(section_name))
        if other_idx != -1 and other_idx < end:
            end = other_idx

    return text[start:end].strip()


# Compare KEY INSIGHTS section across models and scenarios
TARGET_SECTION = "KEY INSIGHTS"
print("=" * 75)
print(f"SIDE-BY-SIDE: '{TARGET_SECTION}' section — executive vs comprehensive")
print("=" * 75)

for model_name in ["gpt-4o-mini", "gpt-5.2"]:
    comp = [r for r in model_results if r["model"] == model_name and r["scenario"] == "comprehensive"]
    exec_ = [r for r in model_results if r["model"] == model_name and r["scenario"] == "executive"]
    if not (comp and exec_):
        continue

    comp_section = extract_section(comp[0]["full_output"], TARGET_SECTION, ALL_SECTIONS)
    exec_section = extract_section(exec_[0]["full_output"], TARGET_SECTION, ALL_SECTIONS)

    print(f"\n{'─'*75}")
    print(f"  {model_name} — EXECUTIVE (focused):")
    print(f"  Length: {len(exec_section)} chars")
    print(f"{'─'*75}")
    print(exec_section[:1500])
    if len(exec_section) > 1500:
        print(f"\n  ... ({len(exec_section) - 1500} more chars)")

    print(f"\n{'─'*75}")
    print(f"  {model_name} — COMPREHENSIVE (full report):")
    print(f"  Length: {len(comp_section)} chars")
    print(f"{'─'*75}")
    print(comp_section[:1500])
    if len(comp_section) > 1500:
        print(f"\n  ... ({len(comp_section) - 1500} more chars)")

    diff = len(exec_section) - len(comp_section)
    pct = (diff / len(comp_section) * 100) if len(comp_section) > 0 else 0
    print(f"\n  → Focused version is {abs(diff)} chars {'longer' if diff > 0 else 'shorter'} ({pct:+.0f}%)")
    print()

In [ ]:
print("=" * 70)
print("TEMPLATE UPLIFT: How much does the template help each model?")
print("=" * 70)
print("(Comparing raw → executive for the same model)\n")

for model_name in MODELS:
    raw_row = df_model[(df_model["model"] == model_name) & (df_model["scenario"] == "raw")]
    exec_row = df_model[(df_model["model"] == model_name) & (df_model["scenario"] == "executive")]
    if raw_row.empty or exec_row.empty:
        continue
    raw_s = raw_row.iloc[0]["sections_found"]
    exec_s = exec_row.iloc[0]["sections_found"]
    raw_c = raw_row.iloc[0]["output_chars"]
    exec_c = exec_row.iloc[0]["output_chars"]
    raw_cost = raw_row.iloc[0]["cost_usd"]
    exec_cost = exec_row.iloc[0]["cost_usd"]

    print(f"  {model_name} ({MODELS[model_name]['label']}):")
    print(f"    Raw:      {raw_s:2d}/11 sections, {raw_c:6d} chars, ${raw_cost:.4f}")
    print(f"    Template: {exec_s:2d}/11 sections, {exec_c:6d} chars, ${exec_cost:.4f}")
    if raw_s > 0:
        print(f"    Uplift:   {exec_s - raw_s:+d} sections ({(exec_s/raw_s - 1)*100:+.0f}%)")
    else:
        print(f"    Uplift:   +{exec_s} sections (raw had 0)")
    print()

In [9]:
from mycontext import Context
from mycontext.foundation import Directive

MODEL = "gpt-4o-mini"


def approx_tokens(text: str) -> int:
    return len(text) // 4


def count_sections_in_output(text: str) -> dict:
    upper = text.upper()
    return {section: section in upper for section in ALL_SECTIONS}


def _extract(result) -> str:
    if hasattr(result, "response"):
        return result.response or ""
    return str(result)


def run_raw() -> dict:
    """Raw: generic prompt, no structured template."""
    prompt = analyzer.generic_prompt(
        data_description=data_description,
        goal=goal,
        context_section=f"\nContext: {context_extra}" if context_extra else "",
    )
    ctx = Context(directive=Directive(content=prompt))
    result = ctx.execute(provider="openai", model=MODEL) if EXECUTE_LLM else "(execute=False)"
    out = _extract(result)
    return {
        "mode": "raw (generic_prompt)",
        "params": {"intent": "N/A", "investment": "N/A"},
        "prompt_chars": len(prompt),
        "output_chars": len(out),
        "output_tokens_approx": approx_tokens(out),
        "sections_found": count_sections_in_output(out),
        "full_output": out,
        "output_preview": out[:1500] + "..." if len(out) > 1500 else out,
    }


def run_intent(intent: str = "comprehensive", investment: str = "standard") -> dict:
    """Run DataAnalyzer with real intent + investment API."""
    ctx = analyzer.build_context(
        data_description=data_description,
        goal=goal,
        context=context_extra,
        intent=intent,
        investment=investment,
    )
    prompt = ctx.directive.content

    if EXECUTE_LLM:
        result = analyzer.execute(
            provider="openai",
            model=MODEL,
            data_description=data_description,
            goal=goal,
            context=context_extra,
            intent=intent,
            investment=investment,
        )
        out = _extract(result)
    else:
        out = "(execute=False)"

    return {
        "mode": f"API (intent={intent})",
        "params": {"intent": intent, "investment": investment},
        "prompt_chars": len(prompt),
        "output_chars": len(out),
        "output_tokens_approx": approx_tokens(out),
        "sections_found": count_sections_in_output(out),
        "full_output": out,
        "output_preview": out[:1500] + "..." if len(out) > 1500 else out,
    }

In [10]:
results = []

print("1/5  Raw (generic_prompt)...")
results.append(run_raw())

print("2/5  API: comprehensive, standard...")
results.append(run_intent(intent="comprehensive", investment="standard"))

print("3/5  API: executive, standard...")
results.append(run_intent(intent="executive", investment="standard"))

print("4/5  API: analyst, quick...")
results.append(run_intent(intent="analyst", investment="quick"))

print("5/5  API: summary, standard...")
results.append(run_intent(intent="summary", investment="standard"))

print("Done — all 5 runs complete.")

1/5  Raw (generic_prompt)...
2/5  API: comprehensive, standard...
3/5  API: executive, standard...
4/5  API: analyst, quick...
5/5  API: summary, standard...
Done — all 5 runs complete.


In [7]:
results

[{'mode': 'raw (generic_prompt)',
  'params': {'intent': 'N/A', 'investment': 'N/A'},
  'prompt_chars': 2462,
  'output_chars': 9807,
  'output_tokens_approx': 2451,
  'sections_found': {'DATA OVERVIEW': False,
   'DESCRIPTIVE STATISTICS': False,
   'PATTERN DETECTION': False,
   'ANOMALY DETECTION': False,
   'CORRELATION ANALYSIS': False,
   'COMPARATIVE ANALYSIS': False,
   'KEY INSIGHTS': True,
   'HYPOTHESES': True,
   'DATA LIMITATIONS': True,
   'RECOMMENDATIONS': False,
   'VISUALIZATION SUGGESTIONS': False},
  'full_output': '## 1) Data summary (type, scope, variables, quality)\n- **Scope/shape:** 36 rows × 6 columns. Monthly observations by **region (North/South/East)** and **product (Widget A/B)**. With 3 regions × 2 products = 6 combos, **36 rows implies ~6 months of data** (assuming complete coverage each month).\n- **Variables:**\n  - Categorical: `month` (object, should be datetime), `region`, `product`\n  - Numeric: `revenue`, `units`, `cost` (COGS)\n- **Quality checks 

## 4. Evidence table: comparison

Metrics: prompt size, output size, token estimate, section coverage.

In [11]:
import pandas as pd

rows = []
for r in results:
    params = r["params"]
    rows.append({
        "mode": r["mode"],
        "intent": params.get("intent", "-"),
        "investment": params.get("investment", "-"),
        "prompt_chars": r["prompt_chars"],
        "output_chars": r["output_chars"],
        "output_tokens_approx": r["output_tokens_approx"],
        "sections_present": sum(r["sections_found"].values()),
    })

df_evidence = pd.DataFrame(rows)
df_evidence

,mode,intent,investment,prompt_chars,output_chars,output_tokens_approx,sections_present
0,raw (generic_prompt),N/A,N/A,2462,4752,1188,3
1,API (intent=comprehensive),comprehensive,standard,4986,7337,1834,11
2,API (intent=executive),executive,standard,2946,3600,900,4
3,API (intent=analyst),analyst,quick,3047,2601,650,5
4,API (intent=summary),summary,standard,2767,3075,768,3


## 5. Section coverage by mode

Which sections appear in each output.

In [12]:
labels = []
for r in results:
    p = r["params"]
    if p.get("intent") != "N/A":
        labels.append(f"{p['intent']} ({p['investment']})")
    else:
        labels.append("raw")
coverage = pd.DataFrame([r["sections_found"] for r in results], index=labels)
coverage.T

,raw,comprehensive (standard),executive (standard),analyst (quick),summary (standard)
DATA OVERVIEW,False,True,True,True,True
DESCRIPTIVE STATISTICS,False,True,False,True,False
PATTERN DETECTION,False,True,False,True,False
ANOMALY DETECTION,False,True,False,False,False
CORRELATION ANALYSIS,False,True,False,True,False
COMPARATIVE ANALYSIS,False,True,False,False,False
KEY INSIGHTS,True,True,True,False,True
HYPOTHESES,False,True,False,False,False
DATA LIMITATIONS,True,True,False,True,False
RECOMMENDATIONS,True,True,True,False,True


## 6. Output previews

First ~1500 chars of each output.

In [13]:
for r in results:
    label = f"{r['mode']}"
    if r["params"].get("intent") != "N/A":
        label += f" (intent={r['params']['intent']}, investment={r['params']['investment']})"
    print("=" * 60)
    print(label)
    print("=" * 60)
    print(r["output_preview"])
    print()

raw (generic_prompt)
### 1. Data Summary

- **Type**: The dataset comprises six columns: `month`, `region`, `product`, `revenue`, `units`, and `cost`. 
- **Scope**: The data spans 36 entries over 12 months for three regions (North, South, East) and two products (Widget A and Widget B).
- **Variables**: 
  - `month`: Represents the time period of data collection.
  - `region`: Denotes the geographical area of sales.
  - `product`: Indicates the type of product sold.
  - `revenue`: Total income generated from sales.
  - `units`: Number of units sold.
  - `cost`: Cost of goods sold (COGS).
- **Quality**: All columns contain non-null entries, indicating no missing values. The data types are appropriate for analysis, with `revenue`, `units`, and `cost` as integers.

### 2. Patterns Identification

- **Monthly Trends**: 
  - Average revenue is around $11,856 with a significant variation (std = $3,456), suggesting fluctuations in sales. 
  - The maximum revenue recorded is $20,100, indicating

## 7. Takeaways (structure & cost)

- **Raw**: Shortest prompt, unstructured output. Section headers depend entirely on the LLM.
- **Comprehensive**: Full structured report using `DataAnalyzer(intent="comprehensive")`. Highest prompt + output size.
- **Executive / Summary / Analyst**: Real `DataAnalyzer` API with `intent` + `investment`. Shorter prompts, focused output, lower token cost.

Quality scores follow in Section 8.

## 8. Quality Test: LLM-as-Judge

Score each of the 5 outputs on four dimensions (1-10):
- **Relevance**: Does the output address the analysis goal?
- **Accuracy**: Are claims supported by the data provided?
- **Actionability**: Are recommendations concrete and usable?
- **Completeness**: Does the output cover what was requested (per its intent)?

Uses a separate LLM call as judge. Each output is scored independently.

In [14]:
import json

JUDGE_PROMPT = """You are an expert data-analysis evaluator. You will receive:
- The DATA that was analyzed
- The GOAL of the analysis
- The INTENT (what type of report was requested)
- The OUTPUT produced by an analysis system

Score the OUTPUT on four dimensions (1-10 scale, 10 = best):

1. **Relevance** (1-10): Does the output address the analysis goal? Is the content on-topic?
2. **Accuracy** (1-10): Are statistical claims, trends, and comparisons supported by the data? Any hallucinations?
3. **Actionability** (1-10): Are recommendations specific, concrete, and usable by the intended audience?
4. **Completeness** (1-10): Given the intent/scope requested, does the output cover everything expected? (A summary should be scored on summary completeness, not full-report completeness.)

Return ONLY valid JSON (no markdown fences) in this exact format:
{"relevance": <int>, "accuracy": <int>, "actionability": <int>, "completeness": <int>, "reasoning": "<1-2 sentence justification>"}
"""


def judge_output(output_text: str, intent: str, data_desc: str, goal_text: str) -> dict:
    """Ask LLM to score a single output on quality dimensions."""
    prompt = (
        f"{JUDGE_PROMPT}\n\n"
        f"---\nDATA:\n{data_desc[:800]}\n\n"
        f"GOAL: {goal_text}\n\n"
        f"INTENT: {intent}\n\n"
        f"OUTPUT:\n{output_text}\n---"
    )
    ctx = Context(directive=Directive(content=prompt))
    result = ctx.execute(provider="openai", model=MODEL, max_tokens=300)
    raw = result.response.strip()
    # Strip markdown fences if the model wraps anyway
    if raw.startswith("```"):
        lines = raw.split("\n")
        raw = "\n".join(lines[1:-1] if len(lines) > 2 else lines)
        raw = raw.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"relevance": 0, "accuracy": 0, "actionability": 0, "completeness": 0, "reasoning": f"Parse error: {raw[:200]}"}


print("Judge function ready.")

Judge function ready.


In [15]:
quality_scores = []

for r in results:
    intent = r["params"].get("intent", "comprehensive")
    label = r["mode"]
    if intent != "N/A":
        label += f" ({intent}, {r['params']['investment']})"

    print(f"Judging: {label}...")
    output_text = r.get("full_output", r["output_preview"])
    print(f"  (judging {len(output_text)} chars of output)")

    scores = judge_output(output_text, intent, data_description, goal)
    scores["mode"] = r["mode"]
    scores["intent"] = intent
    scores["investment"] = r["params"].get("investment", "-")
    scores["label"] = label
    quality_scores.append(scores)
    print(f"  -> R={scores.get('relevance')}, A={scores.get('accuracy')}, Act={scores.get('actionability')}, C={scores.get('completeness')}")

print("\nAll judging complete.")

Judging: raw (generic_prompt)...
  (judging 4752 chars of output)
  -> R=8, A=9, Act=8, C=7
Judging: API (intent=comprehensive) (comprehensive, standard)...
  (judging 7337 chars of output)
  -> R=9, A=8, Act=9, C=9
Judging: API (intent=executive) (executive, standard)...
  (judging 3600 chars of output)
  -> R=10, A=9, Act=10, C=9
Judging: API (intent=analyst) (analyst, quick)...
  (judging 2601 chars of output)
  -> R=9, A=8, Act=8, C=9
Judging: API (intent=summary) (summary, standard)...
  (judging 3075 chars of output)
  -> R=9, A=8, Act=9, C=8

All judging complete.


In [16]:
df_quality = pd.DataFrame(quality_scores)
display_cols = ["label", "relevance", "accuracy", "actionability", "completeness", "reasoning"]
df_quality["avg_score"] = df_quality[["relevance", "accuracy", "actionability", "completeness"]].mean(axis=1).round(1)
df_quality[display_cols + ["avg_score"]]

,label,relevance,accuracy,actionability,completeness,reasoning,avg_score
0,raw (generic_prompt),8,9,8,7,The output effectively addresses the goal of i...,8.0
1,"API (intent=comprehensive) (comprehensive, sta...",9,8,9,9,The output effectively addresses the goal of i...,8.8
2,"API (intent=executive) (executive, standard)",10,9,10,9,The output effectively addresses the goal by p...,9.5
3,"API (intent=analyst) (analyst, quick)",9,8,8,9,The output effectively addresses the goal of i...,8.5
4,"API (intent=summary) (summary, standard)",9,8,9,8,The output effectively addresses the goal of i...,8.5


## 9. Dependency Test: Does hidden context improve insights?

Compare two approaches:
- **Strict**: Ask for KEY INSIGHTS only (no prior sections as context)
- **With hidden context**: Run PATTERN DETECTION first (brief, internal), then feed that into KEY INSIGHTS

If the "with context" version scores higher on quality, the strict approach is risky and we need the hidden-context design. If scores are similar, strict is fine.

In [17]:
MODEL = "gpt-5.2"
context_section = f"\n**CONTEXT**: {context_extra}" if context_extra else ""

# --- Variant A: Strict (KEY INSIGHTS only, no hidden context) ---
strict_directive = f"""Analyze this data:

**DATA**: {data_description}
{context_section}
**ANALYSIS GOAL**: {goal}

Produce ONLY this section:

7. **KEY INSIGHTS**
   - 3-5 findings with evidence, confidence, significance, action.

**OUTPUT FORMAT**: Clear, evidence-based, actionable."""

ctx_strict = Context(directive=Directive(content=strict_directive))
print("Running Strict (key_insights only)...")
result_strict = ctx_strict.execute(provider="openai", model=MODEL, max_tokens=2000)
out_strict = result_strict.response
print(f"  Output: {len(out_strict)} chars, ~{approx_tokens(out_strict)} tokens")

# --- Variant B: Hidden context (run PATTERN DETECTION first, feed into KEY INSIGHTS) ---
pattern_directive = f"""Analyze this data briefly:

**DATA**: {data_description}
{context_section}
**ANALYSIS GOAL**: {goal}

Produce ONLY this section (brief, concise):

3. **PATTERN DETECTION**
   - Trends, seasonality, clusters with evidence.

**CONSTRAINTS**: Be brief. Bullet points only. This is internal context, not a final report."""

ctx_pattern = Context(directive=Directive(content=pattern_directive))
print("\nRunning hidden context pass (pattern detection)...")
result_pattern = ctx_pattern.execute(provider="openai", model=MODEL, max_tokens=1000)
pattern_context = result_pattern.response
print(f"  Pattern context: {len(pattern_context)} chars")

# Now run KEY INSIGHTS with the pattern context injected
enriched_directive = f"""Analyze this data:

**DATA**: {data_description}
{context_section}

**PRIOR ANALYSIS (patterns detected)**:
{pattern_context}

**ANALYSIS GOAL**: {goal}

Using the data AND the prior pattern analysis above, produce ONLY this section:

7. **KEY INSIGHTS**
   - 3-5 findings with evidence, confidence, significance, action.

**OUTPUT FORMAT**: Clear, evidence-based, actionable."""

ctx_enriched = Context(directive=Directive(content=enriched_directive))
print("\nRunning Enriched (key_insights with hidden pattern context)...")
result_enriched = ctx_enriched.execute(provider="openai", model=MODEL, max_tokens=2000)
out_enriched = result_enriched.response
print(f"  Output: {len(out_enriched)} chars, ~{approx_tokens(out_enriched)} tokens")

print("\nDone. Both variants collected.")

Running Strict (key_insights only)...
  Output: 4213 chars, ~1053 tokens

Running hidden context pass (pattern detection)...
  Pattern context: 1812 chars

Running Enriched (key_insights with hidden pattern context)...
  Output: 4210 chars, ~1052 tokens

Done. Both variants collected.


In [18]:
print("Judging Strict (key_insights only)...")
score_strict = judge_output(out_strict, "key_insights_only", data_description, goal)

print("Judging Enriched (key_insights + hidden pattern context)...")
score_enriched = judge_output(out_enriched, "key_insights_with_pattern_context", data_description, goal)

dep_rows = []
for label, scores in [("strict (insights only)", score_strict), ("enriched (insights + hidden patterns)", score_enriched)]:
    scores["label"] = label
    scores["avg_score"] = round(
        (scores.get("relevance", 0) + scores.get("accuracy", 0) +
         scores.get("actionability", 0) + scores.get("completeness", 0)) / 4, 1
    )
    dep_rows.append(scores)

df_dep = pd.DataFrame(dep_rows)
df_dep[["label", "relevance", "accuracy", "actionability", "completeness", "avg_score", "reasoning"]]

Judging Strict (key_insights only)...
Judging Enriched (key_insights + hidden pattern context)...


,label,relevance,accuracy,actionability,completeness,avg_score,reasoning
0,strict (insights only),8,6,8,8,7.5,The insights are largely aligned to growth opp...
1,enriched (insights + hidden patterns),9,6,8,8,7.8,The output provides 5 on-topic insights with c...


In [19]:
print("=" * 60)
print("STRICT — Key Insights Only")
print("=" * 60)
print(out_strict[:2000])
print()
print("=" * 60)
print("ENRICHED — Key Insights with Hidden Pattern Context")
print("=" * 60)
print(out_enriched[:2000])

STRICT — Key Insights Only
7. **KEY INSIGHTS**
- **South + Widget A is the clearest growth engine (prioritize supply + demand capture).**  
  **Evidence:** In the sample, South/Widget A shows the highest January revenue ($15.2k) and strong volume (380 units) versus other region-product pairs (e.g., North/Widget A $12.4k, 310; East/Widget A $9.8k, 245). Given the dataset structure (monthly × region × product) and overall max revenue of **$20.1k**, the top-performing cells are likely concentrated in this segment.  
  **Confidence:** Medium (full table not shown, but visible pattern + max values suggest concentration).  
  **Significance:** High (largest apparent revenue/units pocket; scaling winners is typically highest ROI).  
  **Action:** Ensure inventory/service levels in South for Widget A, expand high-performing channels/accounts, and test modest price increases or premium bundles to lift contribution while monitoring volume elasticity.

- **Widget B appears under-monetized in Sout

## 10. Conclusions

### Quality Test (Section 8)
- Compare `avg_score` across all 5 modes. If parameterized scores are within 1 point of template, quality holds.
- If raw scores noticeably lower, it confirms templates add value.

### Dependency Test (Section 9)
- If **enriched > strict** by 1+ points on avg_score: hidden context matters. Implement the two-pass approach.
- If **enriched ≈ strict**: strict is fine. Ship the simpler design.

### Decision Matrix

| Result | Action |
|--------|--------|
| Parameterized quality ≈ template | Proceed with `intent` + `investment` in DataAnalyzer |
| Parameterized quality << template | Reconsider section dependencies |
| Enriched >> strict | Add hidden context pass for synthesis intents |
| Enriched ≈ strict | Ship strict approach, simpler is better |